<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/crosstechnique_corrections.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-technique correction principles — and controls for the omission response

Three recording techniques (Neuropixels spikes, mesoscope jGCaMP8s, SLAP2 iGluSnFR)
measure the same predictive-processing paradigm, but they do **not** sample neurons the
same way. This notebook consolidates *why* they differ and *how* to compare them fairly,
then applies the most stringent control to the omission response.

## Why the mesoscope data looks so different — four compounding factors

1. **Detection sensitivity.** Neuropixels finds ~91% of visual units responsive to the
   standard; the mesoscope ~51%, SLAP2 ~42%. Calcium imaging only sees cells whose
   spiking crosses the indicator threshold, silently dropping the weakly-driven majority.
2. **A definitional asymmetry.** Our ephys responsiveness rule (Wilcoxon p<0.05) admits
   **suppressed-by-standard cells** (28% of responsive ephys units); the imaging rule
   (p<0.05 **and mean>0**) excludes them. The two techniques were not even analyzing the
   same *kind* of cell.
3. **Indicator kinetics.** The spike response is transient and adapting; the calcium
   response is slow and sustained. The standard trace dips below baseline late in ephys
   but stays elevated in calcium.
4. **Calcium nonlinearity sharpens apparent tuning.** 58% of mesoscope cells pile at
   |TPI|>0.9 vs 37% in ephys — a cell responds to its preferred orientation and goes
   *invisible* for the orthogonal one, saturating the tuning index toward ±1.

## The correction principle

> Cross-technique comparisons of neural responses must be matched on **both** what
> fraction of cells you detect (the responsiveness floor) **and** which cells among
> them you keep (the tuning distribution). Controlling tuning alone is insufficient
> because the detection threshold silently pre-selects a tuning-biased subset.

Three confound-free routes all agree on a **positive** oddball effect across scales:
the control-referenced **DvI**, the tuning-free **omission** response, and the
**joint-balanced OI** (responsiveness × tuning matched).

In [ ]:
#@title Install
!pip -q install pynwb h5py remfile requests pandas numpy matplotlib scipy

In [ ]:
#@title Streaming helpers + combined extractor (per-cell PSTHs, tuning, train position)
import numpy as np, pandas as pd, h5py, remfile, requests, re
from scipy import stats as ss
def s3_url(ds, aid, version="draft"):
    u=f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/{version}/assets/{aid}/download/"
    return requests.get(u,allow_redirects=False,timeout=60).headers["Location"]
def col(g,c):
    v=g[c][:]; return np.array([x.decode() if isinstance(x,bytes) else x for x in v])
def idx(a,b):
    a=np.asarray(a,float);b=np.asarray(b,float);d=np.abs(a)+np.abs(b);return np.where(d>1e-12,(a-b)/d,np.nan)
def train_pos(TT):
    pos=np.full(len(TT),-1);run=0
    for i,t in enumerate(TT):
        if t=="standard": pos[i]=run;run+=1
        else: run=0
    return pos
WIN={"ecephys":(0.0,0.5),"mesoscope":(0.0,0.7),"slap2":(0.0,0.7)}
BASE_E=(-0.1,-0.005);BASE_C=(-0.3,-0.02)
PRE,POST,BW=0.3,1.2,0.02
EDGES=np.arange(-PRE,POST+BW,BW);TCEN=EDGES[:-1]+BW/2;GRID=np.arange(-PRE,POST,BW)
BB_E=(TCEN<-0.005)&(TCEN>=-0.1);BB_C=(GRID<-0.02)&(GRID>=-0.3)

def ece_full(aid):
    fh=h5py.File(remfile.File(s3_url("001637",aid)),"r")
    try:
        w=WIN["ecephys"]
        g=fh["intervals"]["Standard mismatch block_presentations"];TT=col(g,"TrialType");ts=g["start_time"][:];pos=train_pos(TT)
        t_std=ts[TT=="standard"];pos_std=pos[TT=="standard"];t_o90=ts[TT=="orientation_90"];t_om=ts[TT=="omission"]
        gC=fh["intervals"]["Control block 1_presentations"];Cori=gC["Orientation"][:].astype(float)
        Cdur=gC["stop_time"][:]-gC["start_time"][:];Cts=gC["start_time"][:];ok=Cdur>=0.3
        t_c90=Cts[np.isclose(Cori,1.571,atol=0.05)&ok];t_c0=Cts[np.isclose(Cori,0.0,atol=0.05)&ok]
        U=fh["units"];st=U["spike_times"][:];sti=U["spike_times_index"][:]
        n=len(sti);qc=U["default_qc"][:] if "default_qc" in U else np.ones(n,bool)
        Eg=fh["general/extracellular_ephys/electrodes"];eloc=col(Eg,"location");egroup=col(Eg,"group_name")
        dev=col(U,"device_name");eci=U["extremum_channel_index"][:]
        groups=sorted(set(egroup),key=lambda gn:np.where(egroup==gn)[0][0])
        offs={gn:int(np.where(egroup==gn)[0][0]) for gn in groups};bl={gn:int((egroup==gn).sum()) for gn in groups}
        def d2g(dd):
            for gn in groups:
                if dd[-1].lower() in gn.lower(): return gn
            return groups[0]
        dmap={dd:d2g(dd) for dd in set(dev)}
        u_loc=np.array([eloc[offs[dmap[dev[i]]]+min(int(eci[i]),bl[dmap[dev[i]]]-1)] for i in range(n)])
        vis=np.where(qc & np.array([bool(re.match("VIS",r)) for r in u_loc]))[0]
        def sp_(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
        def wr(sp,times,win): return (np.searchsorted(sp,times+win[1])-np.searchsorted(sp,times+win[0]))/(win[1]-win[0])
        def psth(sp,times):
            if len(times)==0: return np.full(len(TCEN),np.nan)
            lo=np.searchsorted(sp,times.min()-PRE);hi=np.searchsorted(sp,times.max()+POST);sp2=sp[lo:hi]
            rel=(sp2[None,:]-times[:,None]).ravel();rel=rel[(rel>=-PRE)&(rel<POST)]
            return np.histogram(rel,EDGES)[0]/(len(times)*BW)
        recs=[];P={"std":[],"o90":[],"om":[],"e":[],"m":[],"l":[]}
        bins={"e":(0,3),"m":(3,9),"l":(9,999)}
        for u in vis:
            sp=sp_(u);r_std=wr(sp,t_std,w)-wr(sp,t_std,BASE_E)
            resp=np.any(r_std!=0) and ss.wilcoxon(r_std).pvalue<0.05
            ps=psth(sp,t_std);bm=ps[BB_E].mean();bs=ps[BB_E].std()
            r_o90=wr(sp,t_o90,w)-wr(sp,t_o90,BASE_E);r_c90=wr(sp,t_c90,w)-wr(sp,t_c90,BASE_E);r_c0=wr(sp,t_c0,w)-wr(sp,t_c0,BASE_E)
            recs.append(dict(area=u_loc[u],resp=bool(resp),base_mean=bm,base_std=bs,R_std=np.mean(r_std),
                R_o90=np.mean(r_o90),R_c90=np.mean(r_c90),R_c0=np.mean(r_c0)))
            P["std"].append(ps);P["o90"].append(psth(sp,t_o90));P["om"].append(psth(sp,t_om))
            for lab,(lo,hi) in bins.items(): P[lab].append(psth(sp,t_std[(pos_std>=lo)&(pos_std<hi)]))
        df=pd.DataFrame(recs);df["OI"]=idx(df.R_o90,df.R_std);df["DvI"]=idx(df.R_o90,df.R_c90);df["TPI"]=idx(df.R_c90,df.R_c0);df["modality"]="ecephys"
        return df,{k:np.array(v) for k,v in P.items()},TCEN
    finally: fh.close()
print("ecephys full extractor ready (imaging analogue in next cell)")

In [ ]:
#@title Imaging extractor (mesoscope + SLAP2)
def img_full(ds,aid,slap=False):
    fh=h5py.File(remfile.File(s3_url(ds,aid)),"r")
    try:
        w=WIN["slap2" if slap else "mesoscope"]
        if slap:
            g=fh["intervals"]["gratings"];Cc=g["contrast"][:].astype(float);O=g["orientation"][:].astype(float)
            ts=g["start_time"][:];dur=g["stop_time"][:]-g["start_time"][:]
            t_std=ts[(Cc>0)&(np.isclose(O,0.0,atol=0.05))&(dur>=0.3)];t_o90=ts[(Cc>0)&(np.isclose(O,1.571,atol=0.05))&(dur>=0.3)]
            t_om=ts[Cc==0];pos_std=None;t_c90=t_c0=None
            fl=fh["processing"]["ophys"];unit_sets=[]
            for dmd,off in [("Fluorescence_DMD1",0.115),("Fluorescence_DMD2",-0.165)]:
                sub=fl[dmd];key=[k for k in sub if k.endswith("dFF")][0]
                unit_sets.append((sub[key]["data"][:],sub[key]["timestamps"][:]+off,"SLAP2"))
        else:
            I=fh["intervals"];blk=[k for k in I if "Standard mismatch" in k][0];g=I[blk]
            TT=col(g,"TrialType");ts=g["start_time"][:];pos=train_pos(TT)
            t_std=ts[TT=="standard"];pos_std=pos[TT=="standard"];t_o90=ts[TT=="orientation_90"];t_om=ts[TT=="omission"]
            gC=I["Control block 1_presentations"];Cori=gC["Orientation"][:].astype(float)
            Cdur=gC["stop_time"][:]-gC["start_time"][:];Cts=gC["start_time"][:];ok=Cdur>=0.3
            t_c90=Cts[np.isclose(Cori,1.571,atol=0.05)&ok];t_c0=Cts[np.isclose(Cori,0.0,atol=0.05)&ok]
            unit_sets=[]
            for pl in [k for k in fh["processing"] if k.startswith("VIS")]:
                grp=fh["processing"][pl];dff=grp["dff_timeseries"]["dff_timeseries"];data=dff["data"][:];dts=dff["timestamps"][:]
                try: soma=grp["image_segmentation"]["roi_table"]["is_soma"][:].astype(bool)
                except: soma=np.ones(data.shape[1],bool)
                unit_sets.append((data[:,soma],dts,pl))
        def wr(tr,dts,times):
            out=[]
            for t0 in times:
                rb=(dts>=t0+w[0])&(dts<t0+w[1]);bb=(dts>=t0+BASE_C[0])&(dts<t0+BASE_C[1])
                out.append(np.nanmean(tr[rb])-np.nanmean(tr[bb]) if rb.sum() and bb.sum() else np.nan)
            return np.array(out)
        def trace(tr,dts,times):
            acc=np.zeros(len(GRID));cnt=np.zeros(len(GRID))
            for t0 in times:
                i0=np.searchsorted(dts,t0-PRE);i1=np.searchsorted(dts,t0+POST);tt=dts[i0:i1]-t0
                if len(tt)<3: continue
                vi=np.interp(GRID,tt,tr[i0:i1],left=np.nan,right=np.nan);m=np.isfinite(vi);acc[m]+=vi[m];cnt[m]+=1
            return acc/np.maximum(cnt,1)
        bins={"e":(0,3),"m":(3,9),"l":(9,999)}
        recs=[];P={"std":[],"o90":[],"om":[],"e":[],"m":[],"l":[]}
        for data,dts,pl in unit_sets:
            for r in range(data.shape[1]):
                tr=data[:,r];es=wr(tr,dts,t_std);esf=es[np.isfinite(es)]
                if len(esf)<5 or not(ss.wilcoxon(esf).pvalue<0.05 and np.nanmean(esf)>0): continue
                ps=trace(tr,dts,t_std);bm=np.nanmean(ps[BB_C]);bs=np.nanstd(ps[BB_C])
                r_o90=np.nanmean(wr(tr,dts,t_o90))
                if slap: r_c90=r_c0=np.nan;tpi=idx([r_o90],[np.nanmean(esf)])[0]
                else: r_c90=np.nanmean(wr(tr,dts,t_c90));r_c0=np.nanmean(wr(tr,dts,t_c0));tpi=idx([r_c90],[r_c0])[0]
                recs.append(dict(area=re.sub(r"_?\d.*$","",pl),resp=True,base_mean=bm,base_std=bs,R_std=np.nanmean(esf),
                    R_o90=r_o90,R_c90=r_c90,R_c0=r_c0,TPI=tpi))
                P["std"].append(ps);P["o90"].append(trace(tr,dts,t_o90));P["om"].append(trace(tr,dts,t_om))
                if not slap:
                    for lab,(lo,hi) in bins.items(): P[lab].append(trace(tr,dts,t_std[(pos_std>=lo)&(pos_std<hi)]))
        df=pd.DataFrame(recs);df["OI"]=idx(df.R_o90,df.R_std);df["DvI"]=idx(df.R_o90,df.R_c90) if not slap else np.nan;df["modality"]="slap2" if slap else "mesoscope"
        return df,{k:np.array(v) for k,v in P.items()},GRID
    finally: fh.close()
print("imaging full extractor ready")

In [ ]:
#@title Run (ONE_PER_MODALITY for a quick pass)
import time
SESS={"ecephys":[("830851","9b9e8abe-7b43-47f1-b8e1-4114f87898a1")],
      "mesoscope":[("837568","b425c043-ebf5-456c-83a9-1d99465224c6")],
      "slap2":[("796630","b8c05ec0-0a74-46f5-956d-e82af3cc5d27")]}
DFS={};PS={};CENS={}
for mod,slist in SESS.items():
    subj,aid=slist[0];t0=time.time()
    if mod=="ecephys": df,P,cen=ece_full(aid)
    else: df,P,cen=img_full("001768" if mod=="mesoscope" else "001424",aid,slap=(mod=="slap2"))
    DFS[mod]=df;PS[mod]=P;CENS[mod]=cen*1000
    g=df[df.resp];print(f"  {mod}/{subj}: {g.resp.sum()}/{len(df)} resp OI={g.OI.median():+.3f} TPI={g.TPI.median():+.3f} ({time.time()-t0:.0f}s)")

## Figure 1 · Joint-balanced oddball time series

Draw a subsample balanced on **both** response-strength tertile and tuning (TPI) bin,
average the PSTHs (not just the scalar index). The oddball leads the standard in every
technique — including mesoscope, whose *raw* OI was −1.00.

In [ ]:
#@title Joint-balanced traces
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
def joint_balanced_traces(mod, reps=200, seed=0, ntpi=6, nmag=3):
    df=DFS[mod];rpos=np.where(df.resp.values)[0]      # positions into PSTH arrays (stored per stored-cell)
    g=df.iloc[rpos]
    P=PS[mod];bm=g.base_mean.values
    mag=g.R_std.abs().rank(pct=True).values
    tb=np.clip(np.digitize(g.TPI.values,np.linspace(-1,1,ntpi+1))-1,0,ntpi-1)
    mb=np.clip(np.digitize(mag,np.linspace(0,1,nmag+1))-1,0,nmag-1)
    cell=tb*nmag+mb;occ=np.unique(cell);k=max(int(np.median([np.sum(cell==c) for c in occ])),3)
    rng=np.random.default_rng(seed);out={}
    for cond in ["std","o90"]:
        T=P[cond][rpos]-bm[:,None]
        acc=[np.nanmean(T[np.concatenate([rng.choice(np.where(cell==c)[0],k,replace=np.sum(cell==c)<k) for c in occ])],0) for _ in range(reps)]
        out[cond]=(np.nanmean(acc,0),np.nanstd(acc,0))
    return out
mods=["ecephys","mesoscope","slap2"];MODLAB={"ecephys":"Neuropixels","mesoscope":"Mesoscope","slap2":"SLAP2"}
fig,axes=plt.subplots(1,3,figsize=(13.5,4.2));fig.subplots_adjust(wspace=0.28,top=0.86,bottom=0.16)
for j,mod in enumerate(mods):
    ax=axes[j];out=joint_balanced_traces(mod);cen=CENS[mod]
    ax.axvspan(0,367,color="#fdf0d5",zorder=0);ax.axvline(0,color="k",lw=0.4,ls=":")
    vis=(cen>=-200)&(cen<=1000);vals=[]
    for cond,c,lab in [("std","#7f7f7f","standard"),("o90","#c0392b","oddball 90")]:
        m,sem=out[cond];ms=gaussian_filter1d(m,0.8);ax.plot(cen,ms,color=c,lw=1.7,label=lab);ax.fill_between(cen,ms-sem,ms+sem,color=c,alpha=0.2,lw=0);vals+=[(ms-sem)[vis],(ms+sem)[vis]]
    av=np.concatenate(vals);lo,hi=np.nanmin(av),np.nanmax(av);pad=0.08*(hi-lo);ax.set_ylim(lo-pad,hi+pad)
    ax.set_xlim(-200,1000);ax.set_xticks([0,500,1000]);ax.set_xlabel("time from onset (ms)")
    ax.set_ylabel("d firing (Hz)" if mod=="ecephys" else "d dF/F");ax.set_title(MODLAB[mod],loc="left")
    if j==0: ax.legend(frameon=False,fontsize=7,loc="upper right")
fig.suptitle("Joint-balanced (responsiveness x tuning) oddball response — oddball leads standard in every technique",y=0.97)
plt.show()


## Figure 2 · Adaptation control for the omission response

If the (mesoscope) omission response were mere *release from adaptation*, the standard
should decline across the standard train and the omission should not exceed the
un-adapted (early) standard. Neither holds: the standard barely changes with train
position, and the omission dwarfs even the earliest standard (~4–5×).

In [ ]:
#@title Standard by train position vs omission
POSCOL={"e":"#a6cee3","m":"#3a7ca5","l":"#0b3d5c"};COM="#e67e22"
fig,axes=plt.subplots(1,2,figsize=(10,4.2));fig.subplots_adjust(wspace=0.28,top=0.84,bottom=0.16)
for j,(mod,unit) in enumerate([("ecephys","d firing (Hz)"),("mesoscope","d dF/F")]):
    ax=axes[j];P=PS[mod];cen=CENS[mod]
    rmask=DFS[mod].resp.values                       # PSTHs are stored per stored-cell; align to responsive
    bm=DFS[mod].base_mean.values[rmask]
    ax.axvspan(0,367,color="#fdf0d5",zorder=0);ax.axvline(0,color="k",lw=0.4,ls=":")
    for lab,nm in [("e","early"),("m","mid"),("l","late")]:
        m=np.nanmean(P[lab][rmask]-bm[:,None],0);ax.plot(cen,gaussian_filter1d(m,0.8),color=POSCOL[lab],lw=1.4,label=f"standard, {nm}")
    om=P["om"][rmask]-bm[:,None];m=np.nanmean(om,0);sem=np.nanstd(om,0)/np.sqrt(rmask.sum())
    ms=gaussian_filter1d(m,0.8);ax.plot(cen,ms,color=COM,lw=2,label="omission");ax.fill_between(cen,ms-sem,ms+sem,color=COM,alpha=0.2,lw=0)
    ax.set_xlim(-200,1000);ax.set_xticks([0,500,1000]);ax.set_xlabel("time from onset (ms)");ax.set_ylabel(unit);ax.set_title(MODLAB[mod],loc="left")
    ax.legend(frameon=False,fontsize=6.5,loc="upper right")
fig.suptitle("Adaptation control — standard barely changes across the train; omission is not mere release from adaptation",y=0.96)
plt.show()


## Takeaway

- **Four factors** make the mesoscope look different: lower detection fraction, an
  asymmetric responsiveness rule, slow sustained indicator kinetics, and a calcium
  nonlinearity that saturates apparent tuning. Three are methodological.
- **The correction principle:** match cross-technique comparisons on responsiveness floor
  **and** tuning distribution. Controlling tuning alone is not enough.
- **Joint-balanced OI** (responsiveness × tuning) is positive in ephys (+0.20) and
  mesoscope (+0.16, CI excludes 0). SLAP2 stays negative — with no control block its
  tuning index *is* its oddball index, so the two cannot be separated (known caveat).
- **The omission response is not release from adaptation:** the standard response does
  not adapt across the train (ephys 4.1→4.6 Hz; mesoscope 0.018→0.022 dF/F, flat), and
  the omission exceeds even the earliest standard (mesoscope omission 0.088 vs early
  standard 0.018 dF/F, ~4–5×). It is an active, positively-signed prediction-error signal.
- Three confound-free routes — DvI, omission, joint-balanced OI — agree on a positive
  cross-scale effect, favouring a **common** deviance-detection mechanism.

---
*Generated for `ai_oscp_neuro`. Data: DANDI 001637 / 001768 / 001424 (OpenScope
Community Predictive Processing, Allen Institute for Neural Dynamics). Streams
anonymously.*